In [1]:
import math
import numpy as np
import random

import time
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.patches import Ellipse
from itertools import chain
import time
from tqdm import tqdm
from operator import attrgetter
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from scipy.optimize import minimize
from scipy.stats import chi2, kurtosis, norm
from scipy.spatial.distance import pdist

In [3]:
# surface functions, merge function, and other helper functions

def surface_single(x, U, A, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for a single point.

    Parameters
    ----------
    x : np.ndarray, shape (d,)
        The query point.
    U : np.ndarray, shape (d, d)
        Eigenvectors (columns) defining the ellipsoid orientation.
    A : np.ndarray, shape (d,)
        Semi-axis lengths along each eigenvector direction.
    p : float or np.ndarray, shape (d)
        power/exponent of the ellipsoid axes.
    k : float
        confidence value

    Returns
    -------
    float
        < 0 if x is inside the ellipsoid
        = 0 if x is on the surface
        > 0 if x is outside the ellipsoid
    """
    x_local = U.T @ x        # Project into ellipsoid's local frame, shape (d,)
    return np.sum(np.abs(x_local / A) ** p) - (k ** 2)

def surface_multi(xs, Us, As, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for n points,
    each against its own ellipsoid.

    Parameters
    ----------
    xs : np.ndarray, shape (n, d)
        Query points.
    Us : np.ndarray, shape (n, d, d)
        Per-ellipsoid eigenvector matrices.
    As : np.ndarray, shape (n, d)
        Per-ellipsoid semi-axis lengths.

    Returns
    -------
    np.ndarray, shape (n,)
        Surface function value for each (point, ellipsoid) pair.
        < 0 inside, = 0 on surface, > 0 outside.
    """
    xs_local = np.einsum('ndi, ni -> nd', Us.swapaxes(1,2), xs)  # (n, d)
    return np.sum(np.abs(xs_local / As) ** p, axis=1) - (k ** 2)              # (n,)

def merge_nodes(x, y): # NOTE: New P is currently avg of the 2 that is (x.p + y.p) / 2
    new_n = x.n + y.n
    dist = x.M - y.M
    new_M = ((x.n * x.M) + (y.n * y.M)) / (new_n)
    new_S = ((x.n / new_n) * x.S) + ((y.n / new_n) * y.S) + (((x.n * y.n)/ (new_n ** 2)) * (np.outer(dist, dist)))
    eigen_value, eigen_vector = np.linalg.eigh(new_S)
    # reverse the order of the eigenvalue/vectors to be in DESCENDING order
    eigen_value = eigen_value[::-1]
    eigen_vector = eigen_vector[:, ::-1]
    # eigen_vector = eigen_vector.T
    # calculate new width that based on confidence ellipsoid
    confidence = new_n / (new_n + x.dim)
    chi = chi2.ppf(confidence, x.dim)
    new_A = np.sqrt(np.abs(eigen_value) * chi)
    new_A[new_A == 0] = x.eps

    return Node(dimension=x.dim, label=x.label, A=new_A, n=new_n, M=new_M, S=new_S, U=eigen_vector, eps=x.eps, alpha=x.alpha, p=(x.p + y.p) / 2)

def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))

def print_time(arr, name):
    print(f"action: {name}")
    print(f"count: {len(arr)}")
    print(f"total: {sum(arr)}")
    print(f"avg: {sum(arr)/len(arr)}")

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]



In [4]:
class Node():
    def __init__(self,
        dimension, label,
        A, p, eps, alpha,
        n=1,
        M=None,
        S=None,
        U=None,
        boundary=None):
        """
        A node contain information about a single ellipsoid such as its centre, cov matrix, and the axis length andd direction.
        It can be "updated" with a point to move it to learn and cover more data point.
        
        Parameters
        A (np.ndarray) (dim) : The length of each axis. Not necessary sorted.  np array 
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function.
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        n (int) : The number of data points this node has learnt from.
        U (np.ndarray) (dim, dim) : The eigenvectors of the cov matrix. NOT transposed that is U[i] contain the ith eigenvector.
        M (np.ndarray) (dim) : The mean/center of the ellipsoid
        S (np.ndarray) (dim, dim) : The cov matrix of the ellipsoid. 
        boundary (dict) : A dictionary containing the boundary point and a special key, _max containing the maximum value.
        max_boundary (int) : The maximum number of points allowed in the boundary.
        """

        self.dim = dimension
        self.label = label
        self.A = A
        self.p = p
        self.eps = eps
        self.alpha = alpha
        self.n = n
        self.confidence = n / (n + self.dim)
        self.chi = chi2.ppf(self.confidence, self.dim)

        self.U = np.eye(dimension) if U is None else U
        self.M = np.array([0.] * dimension) if M is None else M
        self.S = np.zeros((dimension, dimension)) if S is None else S

        if boundary is not None:
            self.boundary = boundary
        else:
            self.boundary = []
        self.max_boundary = 2 * dimension

    def update_boundary(self, x, xd):
        """
        Update the boundary that contain the points nearest to the edge.
        This function is called when the ellipsoid is moved/updated.
        The point x is also considered whether it is a boundary point.
        """
        #print(x, self.boundary)
        # First, update the points in the boundary because the ellipsoid might have moved.
        for p in self.boundary:
            d = surface_single(p - self.M, self.U, self.A, self.p)

            if d > 0:
                for i, x in enumerate(self.boundary):
                    if np.allclose(x, p):
                        self.boundary.pop(i)
                        break

        # If the number of points in the boundary is small we simply add the new point
        d = surface_single(x - self.M, self.U, self.A, self.p)
        if d > 0:
            return None
        if len(self.boundary) < self.max_boundary:
            self.boundary.append(x)
            return None

        # Create copies of the boundary
        A = np.array(self.boundary)
        As = np.repeat(A[None, :, :], A.shape[0], axis=0)
        # Replace one row in each copy of A with x
        As[np.arange(A.shape[0]), np.arange(A.shape[0])] = x

        def f(mats):
            # mats has shape (k, n, m)
            gram = np.matmul(mats.transpose(0, 2, 1), mats)  # shape (k, m, m)
            return np.linalg.det(gram)
        det = f(np.array([A]))
        dets = f(As)

        if det > dets.max():
            pass
        else:
            self.boundary[np.argmax(dets)] = x

        """# Else we remove the point closest to the center (this might be x),        
        if d > self.boundary["_min"][1]:
            self.boundary.pop(self.boundary["_min"][0])
            self.boundary[tuple(x)] = d
            if d > self.boundary["_max"][1]:
                self.boundary["_max"] = (p, d)
            if d < self.boundary["_min"][1]:
                self.boundary["_min"] = (p, d)"""

    def update_exponent(self):
        if len(self.boundary) < self.max_boundary:
            return None
        print("pass")
        x0 = self.p
        cons = []
        for b_point in self.boundary:
            def con(x):
                return - surface_single(b_point - self.M, self.U, self.A, x)
            cons.append({'type': 'ineq', 'fun': con})
            
        solution = minimize(np.sum, x0, method='SLSQP', constraints=cons, bounds=[(1, 10) for _ in range(len(self.p))])
        print("replace",self.p,solution.x)
        self.p = solution.x

    def update(self, x, parent):
        """
        Update the node with a new data point x.
        Performing the following steps.
        1. Calculate the new mean and cov matrix of the node including the new point
        2. Calculate the eigenvalue/vector of the new cov matrix
        3. Update the width of the axes of the ellipsoid
        4. Calculate the growth criterion to check whether the updated ellipsoid cover the new data point
        5. If the growth criterion is acceptable the node is allowed to grow/update to include this new data point,
           else the node is not updated and a new node is added to cover the new datapoint instead.

        Parameter
        x (np.ndarray) (dim) : A new data point the model need to learn/update from.
        parent (VEBF) : The VEBF model this node belong to. We only need this in case we need to add a new node to the VEBF model. 
        """

        # Calculate new mean and cov matrix
        n = self.n
        M = self.M
        alpha = n / (n + 1)
        beta = x / (n + 1)
        M_new = (alpha * M) + beta
        k_1 = (np.outer(x, x)/(n + 1)) - np.outer(M_new, M_new) + np.outer(M, M)
        k_2 = - (np.outer(M, M) / (n + 1))
        k = k_1 + k_2
        S_new = (alpha * self.S) + k

        eigen_value, eigen_vector = np.linalg.eigh(S_new)
        # reverse the order of the eigenvalue/vectors to be in DESCENDING order
        eigen_value = eigen_value[::-1]
        eigen_vector = eigen_vector[:, ::-1]

        # calculate new width with adaptively by combining fixed and dynamic width update rules
        """beta = 1 - self.alpha
        fixed = np.sqrt(np.pi * 2 * np.abs(eigen_value))
        dynamic = self.A + ((M_new - self.M) @ eigen_vector.T)
        new_A = (self.alpha * fixed) + (beta * dynamic)
        new_A[new_A == 0] = self.eps"""

        # calculate new width that based on confidence ellipsoid
        new_A = np.sqrt(np.abs(eigen_value) * self.chi)
        new_A[new_A == 0] = self.eps
        
        gc = surface_single(x - M_new, eigen_vector, self.A, self.p)
        if gc <= 0:
            # update the current node
            self.M = M_new
            self.U = eigen_vector
            self.S = S_new
            self.A = new_A
            self.n += 1
            self.update_boundary(x, gc)
            self.update_exponent()
            return self
        else:
            # add new node
            new_node = Node(self.dim, self.label, M=x, eps=self.eps, A=parent.default_width, alpha=self.alpha, p=self.p)
            parent.nodes[self.label].append(new_node)
            return new_node

    def __str__(self):
        return f"A node covering {self.n} points."

In [5]:
class VEBF():
    def __init__(self, dimension, merge_parameter=0, eps=0.00001, default_width=None, alpha=0.55, p=None):
        """
        The Versatile Ellipsoid Basis Function model.
        Geometrically, the model learn by covering the datapoints with ellipsoids to learn the distribution of the data.
        
        Parameter
        dimension (int) : Number of dimension of input feature.
        merge_parameter (float) : The hyperparameter that controll when two ellipsoids/nodes will be merged.
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function. (dim) np array
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        default_width (np.ndarray) (dim) : The default axis width of the ellipsoid.
        """
        self.dim = dimension
        self.merge_parameter = merge_parameter
        self.nodes = {}
        self.eps = eps
        self.alpha = alpha
        self.default_width = np.array([1.] * dimension) if default_width is None else default_width
        self.p = np.array([2.] * dimension) if p is None else p # default to "normal" hyperellipsoid with p=2

    def find_shortest(self, x, y):
        # TODO: vectorize this
        min_dist = float('inf')
        min_node = None
        for node in self.nodes[y]:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    # NOTE: can combine this with find_shortest by using y=None as default value
    def find_shortest_all(self, x, nodes=None):
        min_dist = float('inf')
        min_node = None
        nodes = self.get_nodes() if nodes is None else nodes
        for node in nodes:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    def check_merge(self, x):
        """
        Check for pair of nodes which can be merged.
        Only consider the "latest" node, x

        Parameters:
        x (Node): The node which has just been updated or created.
        """
        nodes = self.nodes[x.label]
        if len(nodes) == 1:
            return
        nodes.remove(x)
        
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])
        
        scores = surface_multi(x.M - Ms, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(x, y)
            nodes.append(new_node)
            self.check_merge(new_node)
            return

        n = len(nodes)
        U = x.U
        A = x.A
        P = x.p
        Us = np.broadcast_to(U, (n,) + U.shape)
        As = np.broadcast_to(A, (n,) + A.shape)
        Ps = np.broadcast_to(P, (n,) + P.shape)

        scores = surface_multi(Ms - x.M, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(y, x)
            nodes.append(new_node)
            self.check_merge(new_node)
            return
            
        nodes.append(x)

    def get_nodes(self):
        return list(chain.from_iterable(self.nodes.values()))

    def predict(self, x):

        nodes = self.get_nodes()
        assert len(nodes) > 0, "Can't predict with empty VEBF."
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])

        scores = surface_multi(x - Ms, Us, As, Ps)
        idx = np.argsort(scores)
        result = nodes[idx[0]].label
        return result
        """ OLD prediction by euclidean

        candidates = [node for node, score in zip(nodes, scores) if score <= 0]

        if len(candidates) == 0:
            closest = self.find_shortest_all(x)
            return closest.label
        return self.find_shortest_all(x, candidates).label"""
            
    def train(self, x, y):
        if y not in self.nodes: # Add a new node for unseen label.
            self.nodes[y] = [Node(dimension=self.dim, label=y, M=x, eps=self.eps, A=self.default_width, alpha=self.alpha, p=self.p)]
        else: # Update closest existing node for existing label.
            shortest = self.find_shortest(x, y)
            latest = shortest.update(x, self) # latest is the node with latest change, which need to checked for merge
            self.check_merge(latest)

    def __str__(self):
        total = sum(len(x) for x in self.nodes.values())
        string = f"A vebf with {total} nodes.\n"
        for label, nodes in self.nodes.items():
            for node in nodes:
                string += f"Label {label}: {str(node)}\n"
        return string

In [6]:
PMIN = 0.75
PMAX = 6

# Loading the datasets

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]
    
with open("data/iris/iris.data", "r") as f:
    iris_data = f.readlines()
iris_ds = []
for line in iris_data[:-1]:
    line = line.strip().split(',')
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    iris_ds.append((x, y))

with open("data/ecoli/ecoli.data", "r") as f:
    ecoli_data = f.readlines()
ecoli_ds = []
for line in ecoli_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    ecoli_ds.append((x, y))

with open("data/image_seg/segmentation.data", "r") as f:
    seg_train_data = f.readlines()
with open("data/image_seg/segmentation.test", "r") as f:
    seg_test_data = f.readlines()
seg_train = []
for line in seg_train_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_train.append((x, y))
seg_test = []
for line in seg_test_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_test.append((x, y))

with open("data/waveform/waveform.data", "r") as f:
    waveform_data = f.readlines()
wave_ds = []
for line in waveform_data:
    line = line.strip().split(",")
    y = line[-1]
    x = np.array(line[:-1], dtype=float)
    wave_ds.append((x, y))

with open("data/yeast/yeast.data", "r") as f:
    yeast_data = f.readlines()
yeast_ds = []
for line in yeast_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    yeast_ds.append((x, y))

with open("data/anuran/Frogs_MFCCs.csv", "r") as f:
    anuran_data = f.readlines()
anuran_ds = []
for line in anuran_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[:-4], dtype=float)
    y = tuple(line[-4:-1])
    anuran_ds.append((x, y))

with open("data/spambase/spambase.data", "r") as f:
    spam_data = f.readlines()
spam_ds = []
for line in spam_data:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    spam_ds.append((x, y))

with open("data/letter/letter-recognition.data", "r") as f:
    letter_data = f.readlines()
letter_ds = []
for line in letter_data:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    letter_ds.append((x, y))

with open("data/bankrupt/data.csv", "r") as f:
    bankrupt_data = f.readlines()
bankrupt_ds = []
for line in bankrupt_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    bankrupt_ds.append((x, y))

with open("data/digits/optdigits.tes", "r") as f:
    digits_test_data = f.readlines()
digits_test = []
for line in digits_test_data[:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    digits_test.append((x, y))

with open("data/digits/optdigits.tra", "r") as f:
    digits_train_data = f.readlines()
digits_train = []
for line in digits_train_data[:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    digits_train.append((x, y))

with open("data/phishing/PhishingData.arff", "r") as f:
    phishing_data = f.readlines()
phishing_ds = []
for line in phishing_data[14:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    phishing_ds.append((x, y))


In [7]:
def initial_width(ds, delta=1):
    avg_dist = pdist([x for x,_ in ds], metric="euclidean").mean()
    return avg_dist

iris_dim = len(iris_ds[0][0])
anuran_dim = len(anuran_ds[0][0])
spam_dim = len(spam_ds[0][0])
seg_dim = len(seg_train[0][0])
wave_dim = len(wave_ds[0][0])
letter_dim = len(letter_ds[0][0])
yeast_dim = len(yeast_ds[0][0])
digits_dim = len(digits_train[0][0])
bankrupt_dim = len(bankrupt_ds[0][0])
phishing_dim = len(phishing_ds[0][0])

iris_A = initial_width(iris_ds)
anuran_A = initial_width(anuran_ds)
spam_A = initial_width(spam_ds)
seg_A = initial_width(seg_train)
wave_A = initial_width(wave_ds)
letter_A = initial_width(letter_ds)
yeast_A = initial_width(yeast_ds)
digits_A = initial_width(digits_train)
bankrupt_A = initial_width(bankrupt_ds)
phishing_A = initial_width(phishing_ds)
# The implementation of confidence based VEBF takes array of width as initial width
iris_A = np.array([iris_A] * iris_dim)
anuran_A = np.array([anuran_A] * anuran_dim)
spam_A = np.array([spam_A] * spam_dim)
seg_A = np.array([seg_A] * seg_dim)
wave_A = np.array([wave_A] * wave_dim)
letter_A = np.array([letter_A] * letter_dim)
yeast_A = np.array([yeast_A] * yeast_dim)
digits_A = np.array([digits_A] * digits_dim)
bankrupt_A = np.array([bankrupt_A] * bankrupt_dim)
phishing_A = np.array([phishing_A] * phishing_dim)




In [8]:
def plot_superellipsoid_2d(
    C: np.ndarray,
    U: np.ndarray,
    A: np.ndarray,
    P: np.ndarray,
    ax=None,
    resolution: int = 500,
    fill: bool = False,
    fill_alpha: float = 0.25,
    color: str = "steelblue",
    label: str = None,
):
    """
    Visualize a 2-D superellipsoid defined by the implicit function:

        f(x, y) = |u1 · (r - C) / a1|^p1 + |u2 · (r - C) / a2|^p2  ≤  1

    where u1, u2 are the columns of U (axis directions).

    Parameters
    ----------
    C : ndarray, shape (2,)
        Center of the superellipsoid.
    U : ndarray, shape (2, 2)
        Each *column* is a unit vector giving the direction of an axis.
        (Follows the convention where U[:, i] is the i-th axis direction.)
    A : ndarray, shape (2,)
        Half-lengths along each axis.
    P : ndarray, shape (2,)
        Exponents [p1, p2].  p=2 gives an ellipse; p→∞ gives a rectangle;
        0 < p < 2 gives a "pinched" / star shape.
    ax : matplotlib Axes, optional
        Target axes.  A new figure is created when None.
    resolution : int
        Number of grid points per axis for the implicit-function evaluation.
    fill : bool
        Whether to flood-fill the interior.
    fill_alpha : float
        Alpha for the filled region.
    color : str
        Colour used for both the boundary and (if fill=True) the interior.
    label : str, optional
        Legend label.

    Returns
    -------
    ax : matplotlib Axes
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    # ------------------------------------------------------------------
    # 1.  Build a world-space bounding box that safely encloses the shape
    # ------------------------------------------------------------------
    # The extreme world-space extent along each world axis is bounded by
    # the sum of |Uij| * Aj over the body axes j.
    half_extents = np.abs(U) @ A          # shape (2,)
    margin = 0.15 * half_extents.max()

    x_min, x_max = C[0] - half_extents[0] - margin, C[0] + half_extents[0] + margin
    y_min, y_max = C[1] - half_extents[1] - margin, C[1] + half_extents[1] + margin

    xs = np.linspace(x_min, x_max, resolution)
    ys = np.linspace(y_min, y_max, resolution)
    XX, YY = np.meshgrid(xs, ys)

    # ------------------------------------------------------------------
    # 2.  Evaluate the implicit function in the body frame
    # ------------------------------------------------------------------
    # Translate to the centre
    dX = XX - C[0]          # (res, res)
    dY = YY - C[1]

    # Project onto each body axis:  local_i = U[:, i] · (r - C)
    #   U has shape (2, 2); U[:, 0] is first axis direction, etc.
    local = np.stack(
        [U[0, i] * dX + U[1, i] * dY for i in range(2)],
        axis=0,
    )                        # (2, res, res)

    # Normalise by semi-axes and raise to the respective powers
    F = sum(
        (np.abs(local[i]) / A[i]) ** P[i]
        for i in range(2)
    )                        # (res, res)

    # ------------------------------------------------------------------
    # 3.  Draw: filled contour at F=1 (interior) + boundary contour
    # ------------------------------------------------------------------
    if fill:
        ax.contourf(XX, YY, F, levels=[0, 1], colors=[color], alpha=fill_alpha)

    cs = ax.contour(XX, YY, F, levels=[1.0], colors=[color], linewidths=2)

    # Attach a legend proxy if requested
    if label is not None:
        proxy = mpatches.Patch(facecolor=color, alpha=fill_alpha,
                               edgecolor=color, label=label)
        ax.add_patch(plt.Rectangle((0, 0), 0, 0, visible=False))  # dummy
        handles, labels = ax.get_legend_handles_labels()
        handles.append(proxy)
        labels.append(label)
        ax.legend(handles, labels)

    #ax.set_aspect("equal")
    #ax.grid(True, linestyle="--", alpha=0.4)
    return ax

In [9]:
anuran_result = []
for i in tqdm(range(30)):

    # shuffle the train/test split
    anuran_train, anuran_test = train_test_split(anuran_ds)
    model = VEBF(dimension=anuran_dim, merge_parameter=0., default_width=anuran_A)
    for idx in range(len(anuran_train)):
        x,y = anuran_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    # test the model
    correct = 0
    for x, label in anuran_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    anuran_result.append(correct/len(anuran_test))

  0%|          | 0/30 [00:00<?, ?it/s]

pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.05078006 2.00773769 1.65720868 1.60979467 1.94596785 1.37457935
 1.         1.5460237  1.1012519  1.48456075 1.49040085 1.22245692
 1.         1.02559176 1.35906021 1.68882334 1.52713445 2.06048277
 1.7956872  1.00892515 1.60489806 1.22010422]
pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.48706503 1.04423872 1.20747871 1.563699   1.08150389 1.37393129
 1.14867509 1.         1.33648598 1.02448109 1.02753678 1.
 1.61950898 1.         1.00845144 1.46385199 1.14859021 1.34439204
 1.42832456 1.52533036 1.1013933  1.        ]


  3%|▎         | 1/30 [00:01<00:53,  1.86s/it]

pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.54492437 1.23951422 1.40357379 1.36410252 1.38121302 1.0232303
 1.4610853  1.21403189 1.         1.69626929 1.16498219 1.68938198
 1.         1.50702211 1.03503451 1.09943224 1.78619492 1.27763704
 1.36848575 1.03571044 2.03420241 1.29374548]
pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.23732318 1.         1.68421949 1.07693901 1.28475037 1.
 1.47903636 1.42289916 1.54616814 1.23572556 1.         1.
 1.97464438 1.45947099 1.75600143 1.01565212 1.         1.
 1.43580673 1.21110847 1.         1.        ]


  7%|▋         | 2/30 [00:03<00:51,  1.83s/it]

...

 93%|█████████▎| 28/30 [00:57<00:04,  2.08s/it]

pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.         1.23552839 1.55595395 1.74630815 1.         1.37573199
 2.27145255 1.6992785  1.23185225 1.43977453 1.39003161 1.27837384
 1.00879974 2.08628845 1.         1.2389449  1.79378733 1.65245231
 1.44495413 1.65445924 1.58970229 1.        ]
pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [2.18923501 1.43777521 1.         1.98409943 1.98532116 1.32699375
 1.         1.         1.2552055  1.66916889 1.76790168 1.11443894
 1.10100077 1.89638576 1.         1.25038202 2.2881814  1.13721853
 1.37386327 1.52338742 1.60842425 1.61881234]
pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.31008431 1.578877   1.         1.         1.20461014 1.
 1.04424755 1.02686592 1.         1.         1.         1.
 1.10330718 1.21882829 1.30141314 1.13524043 1.75933266 1.35300154
 1.04382262 1.27886873 1.         1.        ]


 97%|█████████▋| 29/30 [00:59<00:02,  2.05s/it]

pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.         1.84600892 2.3913323  1.33486307 1.65973759 1.47038441
 1.15473599 1.11195696 1.68692465 2.12404286 1.6334232  2.90663431
 1.46388816 1.1479598  1.93132931 1.61374308 1.85586873 1.75416655
 1.21396793 1.43947658 1.06233407 1.4738887 ]
pass
replace [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.] [1.52998305 1.17678101 1.51379335 1.2625175  1.44870814 1.
 1.68002125 1.06920102 1.11851492 1.56868395 1.13711725 1.6915517
 1.78894531 1.01941303 1.7049408  1.11102074 1.54951437 1.55532445
 1.45372624 1.         1.84078576 1.26340439]


100%|██████████| 30/30 [01:01<00:00,  2.05s/it]


In [10]:
def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))



In [12]:
print_stat(anuran_result, "anuran")

anuran Mean: 0.727495946258976
anuran Max: 0.8832522585128562
anuran Min: 0.4232105628908965
